In [ ]:
n_pred=(df.final_label=="PREDICTIVE-CANDIDATE").sum()
n_insuf=((df.final_label=="UNCLASSIFIABLE")&(df.reason=="INSUFFICIENT_DATA")).sum()
n_norhy=((df.final_label=="UNCLASSIFIABLE")&(df.reason.str.startswith("NO_FORECASTABLE"))).sum()
n_presc=(df.final_label=="PRESCRIPTIVE (given)").sum()
strong=(df.conf_tier=="Strong").sum(); marg=(df.conf_tier=="Marginal").sum()

print(f"""
============================================================
 RESULT  (of {len(df)} alarms)
------------------------------------------------------------
 PRESCRIPTIVE (given, untouched) ......... {n_presc}
 PREDICTIVE-CANDIDATE .................... {n_pred}
 UNCLASSIFIABLE — INSUFFICIENT_DATA ...... {n_insuf}
 UNCLASSIFIABLE — NO_FORECASTABLE_RHYTHM . {n_norhy}
 label stability: {strong} Strong  /  {marg} Marginal
============================================================

VERDICT — IS THE CURRENT DATA SUFFICIENT?
  PARTIALLY. It is enough to TRIAGE and to FLAG candidates, but NOT
  enough to CONFIRM that an alarm is predictive.
  • "Predictive" = a forecastable rhythm = REGULARITY of the gaps
    between firings. That requires the sequence of individual event
    times. This dataset has only aggregates (count + first/last +
    active-day count), so the best available signal is the active-day
    RATIO — a spread proxy that is necessary but NOT sufficient for
    true forecastability. Hence "PREDICTIVE-CANDIDATE", not "PREDICTIVE".
  • {n_insuf} alarms cannot be judged at all: too few events / too short
    a history to see any pattern.
  • There is NO repair-outcome data, so nothing here can be validated
    against reality — "confidence" is label STABILITY, not accuracy.

WHAT DATA WOULD LET US TRY AGAIN (and actually confirm)
  1. RAW PER-EVENT ALARM LOG (timestamp of every firing, not aggregates).
     -> unlocks true regularity: inter-arrival CV, autocorrelation /
        Ljung-Box, Lomb-Scargle periodicity, spectral entropy. This alone
        upgrades every "candidate" to a measured forecastability score.
  2. IoT / CONDITION-MONITORING SENSOR STREAMS (vibration, motor current
     & torque, temperature, air pressure, cycle counts, line speed).
     -> enables real predictive maintenance: degradation trends and
        remaining-useful-life, which calendar cadence can never give.
  3. MAINTENANCE OUTCOME / RESULTS DATA (did the failure recur after a
     fix? time-to-next-failure, repair effectiveness, PM schedule &
     compliance).
     -> the missing ground truth: turns "stable & sensible" into
        "proven right," and lets us compute real accuracy.
  4. OPERATING CONTEXT (batch / product / recipe, shift, run-state,
     changeovers).
     -> separates process-induced Starved/Blocked/Paused states from
        genuine equipment failures (many top alarms are flow states).
  5. COMPONENT-LEVEL FAILURE MODES (which part failed, finer failure
     codes).

     NEXT STEPS FOR DATA: IOT sensor data, some form of data that shows what the line is doing
     at the occurence. Result data, any result data including if a fix worked or not 
""")

# ---- write results ----
cols=["idap_line","alarm_def_id","alarm_name","top_asset_location","equip_type","frequency_tier",
      "classification","final_label","reason","confidence","conf_tier",
      "occurrence_count","distinct_days_affected","span_days","active_day_ratio",
      "events_per_active_day","mean_gap_days","cm_per_occ","total_downtime_mins"]
out=df[cols].sort_values(["final_label","confidence"],ascending=[True,False])
path=os.path.join(OUT,"idap_predictive_triage.xlsx")
with pd.ExcelWriter(path) as xw:
    out.to_excel(xw,sheet_name="all_alarms",index=False)
    out[out.final_label=="PREDICTIVE-CANDIDATE"].to_excel(xw,sheet_name="predictive_candidates",index=False)
    out[out.final_label=="UNCLASSIFIABLE"].to_excel(xw,sheet_name="unclassifiable",index=False)
print("wrote:",path)

: 